# Instance AI — Workflow-Builder Failure Report

Where the workflow-builder sub-agent is getting stuck, sliced by failure mode.

## Why this exists

The workflow-builder sub-agent runs autonomously inside a sandbox: it writes
TypeScript workflow code, runs `tsc`, and submits via `submit-workflow`. When
it fails, the only signal back to the user is "the agent didn't finish". This
report breaks that opaque category into named, actionable failure modes so we
can prioritize fixes against impact.

## Methodology

**Scope.** Production LangSmith project `instance-ai`, sliding 7-day window.
The unit of analysis is the **conversation thread**: a single user's session
with the agent, possibly spanning many user messages and several builder
invocations.

**Two-layer detection.**

1. **Heuristics (cheap, exhaustive).** Every workflow-builder invocation in
   the window is labelled with zero-or-more violation codes. Heuristics run
   on trace metadata only — no LLM. Thresholds are `p95 of the window ∨ a
   hard floor`, so they auto-adjust to volume but don't dip below
   sanity-checked minimums.
2. **LLM narratives (deep, sampled).** Threads flagged by the heuristics are
   stratified-sampled (≤100), and Opus 4.7 reads each thread's full
   tool-call history to produce a short failure narrative + a short
   `pattern_tag`. A second Opus pass clusters the tags into themes.

The heuristics catch *what* the agent did wrong (loop, thrash, fail). The
LLM narratives explain *why* — the specific node, error message, or contract
the agent kept tripping on.

### Heuristics in this report

| Code | Triggers when builder has… | Floor |
|---|---|---|
| `step_explosion` | ≥ N LLM calls (the loop signal) | 15 |
| `ts_error_loop` | ≥ N `execute_command` (i.e. `tsc`) calls | 5 |
| `edit_thrash` | ≥ N `edit_file` calls | 7 |
| `write_thrash` | ≥ N `write_file` calls | 4 |
| `submit_loop` | ≥ N `submit-workflow` calls | 3 |
| `latency_outlier` | ≥ N seconds wall-clock | 600 |
| `token_outlier` | ≥ N total tokens | 500k |
| `hard_failure` | run errored or `final_status=failed` | — |
| `cancelled` | `final_status=cancelled` | — |
| `builder_retry_in_thread` | thread has ≥ 2 builders, ≥ 1 failed | — |

The dynamic threshold for each metric is shown in the next section.

### LLM sampling — how 100 threads stand in for ~360

Not every flagged thread is sent through Opus (cost, time). We pick threads
by **co-occurrence cluster**: every distinct combination of violation codes
(e.g. `{step_explosion, submit_loop}`) gets representation, capped per
cluster, plus a top-severity tail. The result is breadth across failure
patterns rather than depth on a single one.

## How to read this report

| Section | Question it answers |
|---|---|
| 1. Distributions | Where do builders sit on each metric? Where are the cliffs? |
| 2. Violation counts | How many threads/builders does each heuristic flag? |
| 3. Co-occurrence | Which violations show up together? (Jaccard ≈ 1 → collapse) |
| 4. Top examples per failure mode | The worst offenders, with deep-links into LangSmith |
| 5. Per-thread drilldown | All flagged threads, sortable |
| 6. Worst offenders by metric | Outliers regardless of threshold |
| 7. **LLM-narrated themes** | The "why" — clustered by Opus from the full traces |

## Caveats to keep in mind

- **The LLM sample is stratified, not random.** Theme counts are proportional
  *within the sample*, not the full population.
- **`pattern_tag` per thread is non-deterministic.** Re-running narration may
  produce slightly different tags. The clustered themes are stable.
- **Many "failures" are user-side or infra-side**, not workflow-builder bugs:
  expired OAuth tokens during human approval, 502s from the upstream
  gateway, Cloudflare cancellations, sandbox cold-start timeouts. Treat
  themes accordingly when prioritizing fixes.


In [ ]:
WINDOW = '20260421-20260428'  # set this to the window you want

# LangSmith URL parameters — used to deep-link traces. Override if your tenant differs.
LANGSMITH_TENANT_ID = 'a7b41b9d-bee0-4cf0-9786-f4600380f803'
LANGSMITH_PROJECT_ID = 'cbeb4b95-5d9c-40b4-81f5-6239a427d632'  # instance-ai
LANGSMITH_HOST = 'https://smith.langchain.com'

import json, sys
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

DATA = Path('data/classified') / f'{WINDOW}.json'
blob = json.loads(DATA.read_text())
summary = blob['summary']
builders = pd.DataFrame(blob['builders'])
threads = pd.DataFrame(blob['threads'])

def trace_url(trace_id):
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}/r/{trace_id}?poll=false'

def thread_url(thread_id):
    # Search runs by thread_id metadata; user lands in project filtered to the thread.
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22{thread_id}%5C%22))%22%7D'

print(f'Window:    {summary["window"]}')
print(f'Builders:  {summary["totals"]["builders"]}')
print(f'Threads:   {summary["totals"]["threads"]}')
print(f'Failed:    {summary["totals"]["builders_failed"]}')
print(f'Cancelled: {summary["totals"]["builders_cancelled"]}')
print(f'\nThresholds (p95 ∨ floor):')
for k, v in summary['thresholds'].items():
    print(f'  {k:14s} = {v}')


## TL;DR — top-line numbers

Pulled from the classified + narrated outputs. Refresh by re-running the
notebook against a different `WINDOW`.


In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

narrated_path = Path('data/narrated') / f'{WINDOW}.json'
nb_blob = json.loads(narrated_path.read_text()) if narrated_path.exists() else None

lines = [
    f"- **Window:** `{summary['window']}`",
    f"- **Threads in window:** {summary['totals']['threads']:,}",
    f"- **Builder invocations:** {summary['totals']['builders']:,}",
    f"- **Hard-failed builders:** {summary['totals']['builders_failed']} "
        f"({summary['totals']['builders_failed']/max(1,summary['totals']['builders'])*100:.1f}%)",
    f"- **Cancelled builders:** {summary['totals']['builders_cancelled']} "
        f"({summary['totals']['builders_cancelled']/max(1,summary['totals']['builders'])*100:.1f}%)",
]
if nb_blob:
    lines.append(f"- **LLM-narrated sample:** {nb_blob['sample_size']} threads, {len(nb_blob.get('themes', []))} themes")

display(Markdown("\n".join(lines)))

if nb_blob and nb_blob.get('themes'):
    display(Markdown("### Top failure themes from the LLM pass"))
    th = sorted(nb_blob['themes'], key=lambda x: -x['count'])
    rows = [f"| {t['count']} | **{t['name']}** | {t['description']} |" for t in th]
    display(Markdown(
        "| Threads | Theme | What it means |\n"
        "|--------:|-------|---------------|\n"
        + "\n".join(rows)
    ))


## 1. Distributions

Where do builders sit on the curve? Where are the cliffs? Threshold lines are dashed.

In [ ]:
metrics = [
    ('llm_calls',     'Step count (LLM calls)',          summary['thresholds']['llm_calls']),
    ('latency_s',     'Latency (s)',                     summary['thresholds']['latency_s']),
    ('tokens',        'Total tokens',                    summary['thresholds']['tokens']),
    ('exec_calls',    'mastra_workspace_execute_command', summary['thresholds']['exec_calls']),
    ('edit_calls',    'mastra_workspace_edit_file',      summary['thresholds']['edit_calls']),
    ('write_calls',   'mastra_workspace_write_file',     summary['thresholds']['write_calls']),
    ('submit_calls',  'submit-workflow',                 summary['thresholds']['submit_calls']),
]

from plotly.subplots import make_subplots
fig = make_subplots(rows=4, cols=2, subplot_titles=[m[1] for m in metrics] + [''])
for i, (col, _label, thr) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    vals = builders[col].dropna()
    fig.add_trace(go.Histogram(x=vals, nbinsx=40, marker_color='#5b8def', showlegend=False), row=row, col=c)
    fig.add_vline(x=thr, line_dash='dash', line_color='crimson', row=row, col=c)
fig.update_layout(height=900, title_text='Per-builder metric distributions (red dashed = threshold)', bargap=0.05)
fig.show()

## 2. Violation counts (per thread vs. per builder)

A single bad thread often contains multiple bad builders, so per-thread is the more honest "how many distinct user sessions had this problem" view.

In [ ]:
vc_b = summary['violation_counts_per_builder']
vc_t = summary['violation_counts_per_thread']
all_codes = sorted(set(vc_b) | set(vc_t))

df_v = pd.DataFrame({
    'code': all_codes,
    'per_builder': [vc_b.get(c, 0) for c in all_codes],
    'per_thread': [vc_t.get(c, 0) for c in all_codes],
})
df_v['pct_threads'] = (df_v['per_thread'] / summary['totals']['threads'] * 100).round(1)
df_v = df_v.sort_values('per_thread', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_thread'], orientation='h', name='threads', marker_color='#ef5b5b'))
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_builder'], orientation='h', name='builders', marker_color='#5b8def', opacity=0.6))
fig.update_layout(barmode='overlay', height=420, title='Violations: distinct threads vs total builder events')
fig.show()

display(df_v.sort_values('per_thread', ascending=False).reset_index(drop=True))

## 3. Co-occurrence matrix

Which violations show up together? Cell `(i, j)` = number of threads that triggered both `i` and `j`. Diagonal = total threads with `i`. Strong off-diagonal blocks suggest correlated codes that we may want to collapse into one.

In [ ]:
all_codes_t = sorted(vc_t.keys())
co = pd.DataFrame(0, index=all_codes_t, columns=all_codes_t, dtype=int)
for _, row in threads.iterrows():
    codes = row['violation_codes'] or []
    for a in codes:
        for b in codes:
            if a in co.index and b in co.columns:
                co.loc[a, b] += 1

fig = px.imshow(
    co.values,
    x=co.columns, y=co.index,
    color_continuous_scale='Reds',
    text_auto=True,
    aspect='auto',
    title='Per-thread violation co-occurrence (cell = #threads with both)'
)
fig.update_layout(height=520)
fig.show()

# Jaccard similarity between codes — easier to spot correlated pairs.
import numpy as np
diag = np.diag(co.values).astype(float)
with np.errstate(divide='ignore', invalid='ignore'):
    union = diag[:, None] + diag[None, :] - co.values
    jac = np.where(union > 0, co.values / union, 0.0)
np.fill_diagonal(jac, 0.0)
fig2 = px.imshow(jac, x=co.columns, y=co.index, color_continuous_scale='Blues', text_auto='.2f', aspect='auto',
                title='Jaccard similarity (1.0 = always co-occur)')
fig2.update_layout(height=520)
fig2.show()

## 4. Top examples per failure mode

For each heuristic, the 10 worst builders by the underlying metric. Click the trace_id to open in LangSmith; the thread link filters the project to all runs in that conversation.

In [ ]:
PANEL_BY_CODE = {
    'step_explosion':  'llm_calls',
    'ts_error_loop':   'exec_calls',
    'edit_thrash':     'edit_calls',
    'write_thrash':    'write_calls',
    'submit_loop':     'submit_calls',
    'latency_outlier': 'latency_s',
    'token_outlier':   'tokens',
    'hard_failure':    None,
    'cancelled':       None,
}

builders['violation_codes'] = builders['violations'].apply(
    lambda vs: [v['code'] for v in (vs or [])]
)

for code, sort_col in PANEL_BY_CODE.items():
    sub = builders[builders['violation_codes'].apply(lambda xs: code in xs)].copy()
    if sub.empty:
        continue
    if sort_col:
        sub = sub.sort_values(sort_col, ascending=False)
    sub = sub.head(10)
    rows = []
    for _, b in sub.iterrows():
        rows.append({
            'when': (b['start_time'] or '')[:16].replace('T', ' '),
            'thread': f'<a href="{thread_url(b["thread_id"])}" target="_blank">{(b["thread_id"] or "?")[:8]}</a>' if b['thread_id'] else '?',
            'trace':  f'<a href="{trace_url(b["trace_id"])}" target="_blank">{b["trace_id"][:8]}</a>',
            'llm':    int(b['llm_calls']),
            'exec':   int(b['exec_calls']),
            'edit':   int(b['edit_calls']),
            'write':  int(b['write_calls']),
            'submit': int(b['submit_calls']),
            'latency_s': round(b['latency_s'], 0) if b['latency_s'] else None,
            'tokens': int(b['tokens'] or 0),
            'status': f'{b["status"]} / {b["final_status"]}',
        })
    df = pd.DataFrame(rows)
    display(Markdown(f'### `{code}` — top {len(df)} (sorted by `{sort_col or "recency"}`)'))
    display(HTML(df.to_html(escape=False, index=False)))


## 5. Per-thread drilldown

All threads sorted by violation count, then total tokens. Look here when a specific thread keeps showing up across multiple panels.

In [ ]:
drill = threads.copy()
drill['violation_count'] = drill['violation_codes'].apply(len)
drill['violations'] = drill['violation_codes'].apply(lambda xs: ', '.join(xs))
drill['link'] = drill['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{t[:8]}</a>')
drill['first'] = drill['first_start'].str[:16].str.replace('T', ' ')
drill = drill[['link', 'first', 'builder_count', 'fail_count', 'total_tokens', 'violation_count', 'violations']]
drill = drill.rename(columns={'link': 'thread'})
drill = drill.sort_values(['violation_count', 'total_tokens'], ascending=[False, False])

display(Markdown(f'**{len(drill)} threads, top 30 shown**'))
display(HTML(drill.head(30).to_html(escape=False, index=False)))

## 6. Worst offenders by metric

Top builders for each individual metric, regardless of whether they crossed a threshold. Useful for spotting near-misses or extreme outliers.

In [ ]:
def top_by(col, n=10):
    sub = builders.sort_values(col, ascending=False).head(n).copy()
    sub['trace'] = sub['trace_id'].apply(lambda t: f'<a href="{trace_url(t)}" target="_blank">{t[:8]}</a>')
    sub['thread'] = sub['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{(t or "?")[:8]}</a>' if t else '?')
    sub['violations'] = sub['violation_codes'].apply(lambda xs: ', '.join(xs))
    keep = ['thread', 'trace', col, 'llm_calls', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls', 'latency_s', 'tokens', 'status', 'final_status', 'violations']
    return sub[[c for c in keep if c in sub.columns]]

for col in ['llm_calls', 'latency_s', 'tokens', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls']:
    display(Markdown(f'### Top by `{col}`'))
    display(HTML(top_by(col).to_html(escape=False, index=False)))

## 7. LLM-narrated failure clusters

Output of `narrate.py` — Opus 4.7 reads each sampled thread's full I/O,
produces a structured failure narrative, then a second pass clusters
the narratives into themes. Loads `data/narrated/<window>.json` if it
exists; skip this section otherwise.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML

narrated_path = Path('data/narrated') / f'{WINDOW}.json'
if not narrated_path.exists():
    display(Markdown(f"_No narrated data at `{narrated_path}` — run `./narrate.py --window {WINDOW}` to generate._"))
else:
    nb_blob = json.loads(narrated_path.read_text())
    nar_by_thread = {n['thread_id']: n for n in nb_blob['narratives']}
    themes = nb_blob.get('themes', [])

    display(Markdown(
        f"**Sample:** {nb_blob['sample_size']} threads &nbsp;|&nbsp; "
        f"**Model:** {nb_blob['model']} &nbsp;|&nbsp; "
        f"**Tokens:** {nb_blob['tokens']['input']:,} in / {nb_blob['tokens']['output']:,} out"
    ))

    # Theme overview chart
    if themes:
        import plotly.express as px
        df_themes = pd.DataFrame([
            {'theme': t['name'], 'count': t['count']} for t in themes
        ]).sort_values('count', ascending=True)
        fig = px.bar(df_themes, x='count', y='theme', orientation='h',
                     title=f"Themes across {nb_blob['sample_size']} sampled threads",
                     text='count')
        fig.update_layout(height=max(360, 36 * len(themes)))
        fig.show()

    # Per-theme drilldown
    for t in sorted(themes, key=lambda x: -x['count']):
        ex_links = []
        for tid in t.get('example_thread_ids', [])[:5]:
            n = nar_by_thread.get(tid)
            tag = n['narrative']['pattern_tag'] if n else '?'
            ex_links.append(f'<a href="{thread_url(tid)}" target="_blank"><code>{tid[:8]}</code></a> <em>{tag}</em>')
        display(Markdown(
            f"### {t['name']} &nbsp;<small>({t['count']} threads)</small>\n\n"
            f"{t['description']}\n\n"
            f"**Pattern tags:** " + ', '.join(f'`{tg}`' for tg in t.get('pattern_tags', [])) + "\n\n"
            f"**Examples:** " + ' &nbsp;·&nbsp; '.join(ex_links)
        ))


### All sampled narratives

Sortable table of every thread we sent through the LLM.


In [ ]:
rows = []
for n in nb_blob['narratives']:
    nv = n.get('narrative') or {}
    rows.append({
        'thread': f'<a href="{thread_url(n["thread_id"])}" target="_blank">{n["thread_id"][:8]}</a>',
        'tag': nv.get('pattern_tag', '?'),
        'codes': ', '.join(n.get('violation_codes', [])),
        'intent': nv.get('user_intent', ''),
        'failure': nv.get('failure_narrative', ''),
        'conf': nv.get('confidence', ''),
    })
df_n = pd.DataFrame(rows)
display(HTML(df_n.to_html(escape=False, index=False, max_rows=None)))
